In [1]:
# 兼容 Jupyter 已存在事件循环，避免 Engine.generate/encode 抛出 "This event loop is already running"
import importlib

if importlib.util.find_spec("nest_asyncio") is not None:
    import nest_asyncio
    nest_asyncio.apply()
    print("[patch] nest_asyncio applied")
else:
    print("[patch] nest_asyncio not found; skip")

[patch] nest_asyncio applied


# SGLang 注意力后端配置入门：统一后端与分离后端最小实践

**定位**：这是一篇入门向 SGLang Notebook，聚焦 `sgl.Engine(...)` 的内核后端配置参数（`attention_backend`、`prefill_attention_backend`、`decode_attention_backend`、`disable_flashinfer_autotune`），帮助你先跑通、再做最小对比实验。

**选题来源（UT -> 教程化）**：本主题参考了 `sglang/test/srt/cpu/test_flash_attn.py`、`sglang/sgl-kernel/tests/test_flash_attention.py`、`sglang/sgl-kernel/tests/test_flash_attention_4.py` 中“多后端/不同注意力路径可切换与校验”的思路，但本教程只保留 Python 入门可复现路径，不涉及底层 CUDA/Triton 内核开发。

**开源说明**：本文为原创教学示例，可用于学习与社区分享（如 Gitee）；请同时遵守所使用模型与第三方依赖的许可证。

---

## 版本说明（请先确认）

当前仓库未提供 `community_materials_nb/sglang/SGLANG_NOTEBOOK_FACTS.md`，因此本教程按“通用、入门、保守参数”编写。

建议你先运行下面命令确认本机 SGLang 版本，再对照官方文档：

```bash
python -c "import sglang as sgl; print(getattr(sgl, '__version__', 'unknown'))"
```

官方文档：<https://docs.sglang.io/>

---

## 适用显卡与环境建议

| 项目 | 建议 |
|---|---|
| 显卡类型 | NVIDIA GPU（单卡即可入门） |
| 建议显存 | 8GB 及以上（更推荐 12GB+） |
| 驱动 / CUDA | 与本机 `torch` 匹配（建议先在自检单元确认） |
| Python | 3.8+ |
| 系统 | Linux 优先（Windows 建议 WSL2） |
| 默认模型 | `Qwen/Qwen2.5-1.5B-Instruct` |

> 说明：若显存偏小，可优先使用 `torch_native` 后端并降低 `max_total_tokens`。

---

## 完整依赖安装命令（终端执行）

```bash
# 0) 创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate

# 1) 升级 pip
python -m pip install --upgrade pip

# 2) 按你的 CUDA 版本安装 torch（下面仅示例）
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3) 安装 SGLang + OpenAI SDK
pip install -U sglang openai

# 4) 运行本 Notebook 需要 Jupyter（任选其一）
pip install jupyter

# 5) （可选）若你也想复现实验中涉及的 UT 张量处理辅助库
pip install -U einops
```

> 说明：本教程依赖与 UT 主题相关的核心包主要是 `torch`；`einops` 仅在你进一步复现实验测试脚本时可选。

---

## 简单使用说明

1. 先在终端执行上面的安装命令，并激活虚拟环境。
2. 在本jupyter notebook所在路径下，启动 Jupyter：jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
3. 按顺序运行下面单元格（环境自检 -> 统一后端实验 -> 分离后端实验 -> 资源释放）。
4. 首次运行会联网下载模型，速度取决于网络环境。

---

In [2]:
# 环境自检：Python / torch / CUDA / sglang 版本

import sys  # 读取 Python 版本

print("Python:", sys.version)

try:
    import torch  # 检查 torch 是否可导入
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print("GPU memory (GB):", round(total_gb, 2))
except Exception as e:
    print("torch 检查失败:", repr(e))

try:
    import sglang as sgl  # 检查 sglang 是否可导入
    print("sglang:", getattr(sgl, "__version__", "unknown"))
except Exception as e:
    print("sglang 检查失败:", repr(e))

Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


torch: 2.9.1+cu129
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
GPU memory (GB): 79.18


sglang: 0.5.9


## 概念速读：本篇要观察的 4 个参数

- `attention_backend`：统一指定注意力后端（例如 `triton` / `flashinfer` / `fa3` / `torch_native`）。
- `prefill_attention_backend` 与 `decode_attention_backend`：分别指定 prefill 与 decode 的后端，优先级高于统一后端。
- `disable_flashinfer_autotune`：是否关闭 FlashInfer 自动调优；用于观察首轮 warmup 行为差异时很实用。
- 由于不同硬件支持不同后端，本教程采用“候选后端 + 自动回退”方式，尽量提升首次跑通成功率。

---

In [3]:
# Demo 1：统一 attention_backend，带自动回退

import sglang as sgl  # 导入 SGLang
import torch  # 检查是否可用 CUDA

# 关键常量集中定义，便于新手只改这里
MODEL_PATH = "Qwen/Qwen2.5-1.5B-Instruct"  # 入门小模型
TP_SIZE = 1  # 单卡默认 1
MEM_FRACTION_STATIC = 0.70  # 静态显存占比
MAX_RUNNING_REQUESTS = 8  # 并发请求上限
MAX_TOTAL_TOKENS = 8192  # token 池上限
CHUNKED_PREFILL_SIZE = 1024  # prefill chunk 大小
DISABLE_FLASHINFER_AUTOTUNE = True  # 关闭 FlashInfer 自动调优（便于观察）

# 根据设备准备候选后端；若无 CUDA，优先走 torch_native
if torch.cuda.is_available():
    backend_candidates = ["triton", "flashinfer", "fa3", "torch_native"]
else:
    backend_candidates = ["torch_native"]


def create_engine_with_backend_fallback(candidates):
    """按候选顺序尝试初始化 Engine，直到成功。"""
    last_error = None
    for backend in candidates:
        try:
            eng = sgl.Engine(
                model_path=MODEL_PATH,
                tp_size=TP_SIZE,
                mem_fraction_static=MEM_FRACTION_STATIC,
                max_running_requests=MAX_RUNNING_REQUESTS,
                max_total_tokens=MAX_TOTAL_TOKENS,
                chunked_prefill_size=CHUNKED_PREFILL_SIZE,
                attention_backend=backend,
                disable_flashinfer_autotune=DISABLE_FLASHINFER_AUTOTUNE,
            )
            print("[OK] 已使用统一后端:", backend)
            return eng, backend
        except Exception as e:
            print("[跳过] 后端不可用:", backend, "|", repr(e))
            last_error = e

    raise RuntimeError(f"所有候选后端都不可用，最后错误: {repr(last_error)}")


# 初始化引擎
engine, selected_unified_backend = create_engine_with_backend_fallback(backend_candidates)

# 准备输入
prompts = [
    "请用三句话解释 attention backend 在推理中的作用。",
    "请解释 prefill 与 decode 阶段为什么可能使用不同后端。",
]

sampling_params = {
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 96,
}

# 执行推理
outputs = engine.generate(prompts, sampling_params)

# 打印结果
print("\n[统一后端实验] selected_backend =", selected_unified_backend)
for i, out in enumerate(outputs):
    text = out.get("text", str(out)) if isinstance(out, dict) else str(out)
    print(f"\n===== 样本 {i} =====")
    print(text[:800])

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[2026-04-06 09:12:07] INFO engine.py:156: server_args=ServerArgs(model_path='Qwen/Qwen2.5-1.5B-Instruct', tokenizer_path='Qwen/Qwen2.5-1.5B-Instruct', tokenizer_mode='auto', tokenizer_worker_num=1, skip_tokenizer_init=False, load_format='auto', model_loader_extra_config='{}', trust_remote_code=False, context_length=None, is_embedding=False, enable_multimodal=None, revision=None, model_impl='auto', host='127.0.0.1', port=30000, fastapi_root_path='', grpc_mode=False, skip_server_warmup=False, warmups=None, nccl_port=None, checkpoint_engine_wait_weights_before_ready=False, dtype='auto', quantization=None, quantization_param_path=None, kv_cache_dtype='auto', enable_fp32_lm_head=False, modelopt_quant=None, modelopt_checkpoint_restore_path=None, modelopt_checkpoint_save_path=None, modelopt_export_path=None, quantize_and_serve=False, rl_quant_profile=None, mem_fraction_static=0.7, max_running_requests=8, max_queued_requests=None, max_total_tokens=8192, chunked_prefill_size=1024, enable_dynami

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.54it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.54it/s]



Capturing batches (bs=8 avail_mem=24.44 GB):   0%|          | 0/4 [00:00<?, ?it/s]

Capturing batches (bs=1 avail_mem=24.37 GB): 100%|██████████| 4/4 [00:20<00:00,  5.05s/it]


[OK] 已使用统一后端: triton



[统一后端实验] selected_backend = triton

===== 样本 0 =====
 Attention backend 在推理中的作用是通过注意力机制，根据输入数据的局部相关性来调整模型的计算，从而提高模型的计算效率和准确性。它通过计算注意力权重，将模型的计算资源集中在输入数据的局部区域，从而减少不必要的计算，提高模型的计算效率。同时，注意力机制还可以根据输入数据的局部相关性，对模型的输出进行调整，从而提高模型的准确性。总之，注意力机制在推理中起到了

===== 样本 1 =====
 在使用预填充（prefill）和解码（decode）阶段时，选择不同的后端（backend）可能会带来不同的性能和资源使用情况。以下是一些可能的原因：

1. **预填充阶段**：
   - **性能**：预填充阶段通常涉及大量的数据处理和计算，因此选择一个高效的后端可以显著提高性能。例如，使用GPU加速的后端可以显著提高预填充阶段的计算速度。
   - **


In [4]:
# Demo 2（可选）：分离 prefill / decode 后端，带自动回退

import sglang as sgl  # 导入 SGLang

# 若你单独运行这个 cell，则补一份默认常量
MODEL_PATH = globals().get("MODEL_PATH", "Qwen/Qwen2.5-1.5B-Instruct")
TP_SIZE = globals().get("TP_SIZE", 1)
MEM_FRACTION_STATIC = globals().get("MEM_FRACTION_STATIC", 0.70)
MAX_RUNNING_REQUESTS = globals().get("MAX_RUNNING_REQUESTS", 8)
MAX_TOTAL_TOKENS = globals().get("MAX_TOTAL_TOKENS", 8192)
CHUNKED_PREFILL_SIZE = globals().get("CHUNKED_PREFILL_SIZE", 1024)
DISABLE_FLASHINFER_AUTOTUNE = globals().get("DISABLE_FLASHINFER_AUTOTUNE", True)

# 候选组合：先试“分离组合”，再回退到更保守组合
split_backend_candidates = [
    ("fa3", "triton"),
    ("triton", "triton"),
    ("torch_native", "torch_native"),
]

split_engine = None
selected_split = None
last_error = None

for prefill_backend, decode_backend in split_backend_candidates:
    try:
        split_engine = sgl.Engine(
            model_path=MODEL_PATH,
            tp_size=TP_SIZE,
            mem_fraction_static=MEM_FRACTION_STATIC,
            max_running_requests=MAX_RUNNING_REQUESTS,
            max_total_tokens=MAX_TOTAL_TOKENS,
            chunked_prefill_size=CHUNKED_PREFILL_SIZE,
            prefill_attention_backend=prefill_backend,
            decode_attention_backend=decode_backend,
            disable_flashinfer_autotune=DISABLE_FLASHINFER_AUTOTUNE,
        )
        selected_split = (prefill_backend, decode_backend)
        print("[OK] 已使用分离后端组合:", selected_split)
        break
    except Exception as e:
        print("[跳过] 分离后端不可用:", (prefill_backend, decode_backend), "|", repr(e))
        last_error = e

if split_engine is not None:
    split_prompt = [
        "请简要说明为什么 decode 阶段通常更关注低延迟。",
    ]
    split_sampling_params = {
        "temperature": 0.2,
        "top_p": 0.9,
        "max_new_tokens": 80,
    }
    split_outputs = split_engine.generate(split_prompt, split_sampling_params)
    print("\n[分离后端实验] selected_split =", selected_split)
    print(split_outputs[0].get("text", str(split_outputs[0]))[:800])
else:
    print("\n当前环境没有可用的分离后端组合。")
    print("最后错误:", repr(last_error))
    print("建议：保留 Demo 1 的统一后端方案先完成学习路径。")

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.44it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.44it/s]



Capturing batches (bs=8 avail_mem=20.24 GB):   0%|          | 0/4 [00:00<?, ?it/s]

Capturing batches (bs=1 avail_mem=20.17 GB): 100%|██████████| 4/4 [00:00<00:00,  6.42it/s]


[OK] 已使用分离后端组合: ('fa3', 'triton')



[分离后端实验] selected_split = ('fa3', 'triton')
 由于 decode 阶段通常涉及解码视频或音频数据，因此它需要尽可能快地处理数据以保持流媒体或视频会议等实时应用的流畅性。如果解码延迟过高，会导致视频或音频质量下降，甚至出现卡顿或丢帧等问题，从而影响用户体验。因此，decode 阶段通常更关注低延迟，以确保视频


In [5]:
# 释放资源：建议实验结束后执行

# 依次关闭可能存在的引擎对象，避免显存持续占用
for name in ["split_engine", "engine"]:
    obj = globals().get(name, None)
    if obj is None:
        continue

    shutdown_fn = getattr(obj, "shutdown", None)
    if callable(shutdown_fn):
        shutdown_fn()
        print(f"{name}.shutdown() 已执行")
    else:
        print(f"{name} 未暴露 shutdown()，可直接重启 kernel 释放资源")

split_engine.shutdown() 已执行
engine.shutdown() 已执行


## 常见报错与处理（1~2 条）

1. **OOM / CUDA out of memory**
   - 降低 `mem_fraction_static`（如 `0.70 -> 0.60`）并下调 `max_total_tokens`（如 `8192 -> 4096`）。
   - 改用更保守后端（先 `torch_native`），并减少并发请求数量。

2. **后端初始化失败（某 backend 不可用）**
   - 这是常见现象：不同 GPU 架构与驱动对后端支持不同。
   - 保留“自动回退”逻辑，优先保证至少一个后端可跑通，再做对比实验。

---

如果你希望，我可以再给这份 notebook 追加一个“统一后端 vs 分离后端”的简单耗时统计单元格（只做入门级观察，不做严格 benchmark）。